<a href="https://colab.research.google.com/github/yoshiagent/linearControl-colab/blob/main/notebooks/StateSpace_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 狀態空間設計 State-Space Design

對照網頁：[線性控制｜狀態空間設計](https://yoshiagent.github.io/linearControl/SS.html)

原課程以 MATLAB 示範狀態空間的標準式轉換、極點配置（regulator 與 type 1 servo）與全階觀測器。
這本筆記本先列出原本的 MATLAB 程式，再給逐段對應的 Python 版本，並用手算的特徵多項式驗證增益。

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `phasevariable.m` | 用可控性矩陣把任意實現轉成相變數（可控）標準式 | 第 1 節 |
| `poleplaceregu.m` | Pole Placement — Regulator：$u=-Kx$，初始狀態回到零 | 第 2 節 |
| `poleplace.m` | Pole Placement — Type 1 Servo：受控體含積分器，$u=-Kx+k_1r$ | 第 3 節 |
| `observer.m` | 全階觀測器增益 $K_e$（對偶：`place(A',C',p)'`） | 第 4 節 |
| `observerregulator.m` | 觀測器＋狀態回授（倒單擺線性化模型） | 第 5 節 |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal, linalg
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def clean(M, d=4):
    """四捨五入並去掉 -0.0，方便列印"""
    return (np.round(np.real_if_close(np.asarray(M, dtype=complex)), d) + 0.0).tolist()


def ctrb(A, B):
    """仿 MATLAB ctrb：M = [B AB A^2B …]"""
    n = A.shape[0]
    return np.hstack([np.linalg.matrix_power(A, i) @ B for i in range(n)])


def obsv(A, C):
    """仿 MATLAB obsv：O = [C; CA; CA^2; …]"""
    n = A.shape[0]
    return np.vstack([C @ np.linalg.matrix_power(A, i) for i in range(n)])


def place(A, B, p):
    """仿 MATLAB place：求 K 使 eig(A-BK) = p（scipy.signal.place_poles）"""
    return np.real(signal.place_poles(A, B, p).gain_matrix)


def damp(A):
    """仿 MATLAB damp：每個極點的 (極點, wn, zeta)"""
    return [(p, abs(p), -p.real / abs(p)) for p in sorted(np.linalg.eigvals(A), key=lambda z: (z.real, z.imag))]


def initial(A, x0, t):
    """零輸入響應 x(t) = e^{At} x0（MATLAB 原程式用 lsim(sys, 0, t, x0)）"""
    return np.array([linalg.expm(A * ti) @ x0 for ti in t])

## 1. 相變數標準式 `phasevariable.m`

```matlab
syszpk=zpk([],[-3 -1 -2],1)
sysss=ss(syszpk)
[A B C D]=ssdata(sysss)
M=[B A*B A^2*B]
JA=poly(A)
a1=JA(2)
a2=JA(3)
W=[a2 a1 1;a1 1 0;1 0 0]
T=M*W
AA=inv(T)*A*T
BB=inv(T)*B
```

受控體 $G(s)=\dfrac{1}{(s+1)(s+2)(s+3)}=\dfrac{1}{s^3+6s^2+11s+6}$。任一組狀態實現 $(A,B)$ 只要可控，
令 $T = MW$（$M$ 為可控性矩陣，$W$ 由特徵多項式係數 $a_1,a_2$ 組成），$\bar A = T^{-1}AT$、$\bar B=T^{-1}B$ 就是相變數標準式：

$$\bar A=\begin{bmatrix}0&1&0\\0&0&1\\-a_3&-a_2&-a_1\end{bmatrix},\qquad \bar B=\begin{bmatrix}0\\0\\1\end{bmatrix}$$

MATLAB 的 `ss(zpk)` 與 scipy 的 `zpk2ss` 產生的實現不同（狀態變數選法不同），但轉換後的標準式應該一樣。
下面另用一組刻意打亂的實現（相似轉換）再驗證一次。

In [2]:
A, B, C, D = signal.zpk2ss([], [-3, -1, -2], 1)
print("scipy zpk2ss realization:\nA =\n", A, "\nB =", B.ravel())


def phase_variable(A, B):
    n = A.shape[0]
    M = ctrb(A, B)
    JA = np.poly(A)                         # [1, a1, a2, a3]
    W = np.zeros((n, n))
    for i in range(n):                      # W = [a2 a1 1; a1 1 0; 1 0 0]（n = 3 時）
        for j in range(n - i):
            W[i, j] = JA[n - 1 - i - j] if i + j < n - 1 else 1.0
    T = M @ W
    return np.linalg.solve(T, A @ T), np.linalg.solve(T, B), JA, W


AA, BB, JA, W = phase_variable(A, B)
print("JA = poly(A) =", clean(JA))
print("W =", clean(W))
print("AA =", clean(AA))
print("BB =", clean(BB.ravel()))

rng = np.random.default_rng(1)
P = rng.normal(size=(3, 3))                 # 任意可逆矩陣：另一組狀態實現
A2, B2 = np.linalg.solve(P, A @ P), np.linalg.solve(P, B)
AA2, BB2, _, _ = phase_variable(A2, B2)
print("another realization -> AA =", clean(AA2), " BB =", clean(BB2.ravel()))

scipy zpk2ss realization:
A =
 [[ -6. -11.  -6.]
 [  1.   0.   0.]
 [  0.   1.   0.]] 
B = [1. 0. 0.]
JA = poly(A) = [1.0, 6.0, 11.0, 6.0]
W = [[11.0, 6.0, 1.0], [6.0, 1.0, 0.0], [1.0, 0.0, 0.0]]
AA = [[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [-6.0, -11.0, -6.0]]
BB = [0.0, 0.0, 1.0]
another realization -> AA = [[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [-6.0, -11.0, -6.0]]  BB = [0.0, 0.0, 1.0]


## 2. Pole Placement — Regulator `poleplaceregu.m`

```matlab
a=[0 1 0;0 0 1;-6 -11 -6];
b=[0 0 10]';
c=[1 0 0];
d=0;
sys=ss(a,b,c,d);
t=0:0.01:5;
u=zeros(1,length(t));
[y,t,x] = lsim(sys,u,t,x0);
figure(1)
plot(t,x)
grid
p=[-2+2*sqrt(3)*i -2-2*sqrt(3)*i -10];
k=place(a,b,p);
aa=a-b*k;
bb=[0 0 0]';
x0=[1 0 0];
sys1=ss(aa,bb,c,d)
[y1,t,x1] = lsim(sys1,u,t,x0);
figure(2)
plot(t,x1)
grid
```

控制律 $u=-Kx$，閉迴路 $\dot x=(A-BK)x$，解為 $x(t)=e^{(A-BK)t}x(0)$；$A-BK$ 的特徵值稱為 regulator poles。
只要可控（$\operatorname{rank}M=n$），極點就能任意配置。

> **編註**：原程式第一次呼叫 `lsim(sys,u,t,x0)` 時 `x0` 還沒定義（定義在後面），直接執行會出錯；這裡把 `x0=[1 0 0]` 移到前面。

手算驗證：$A-BK$ 的最後一列是 $[-6-10k_1,\ -11-10k_2,\ -6-10k_3]$，特徵多項式 $s^3+(6+10k_3)s^2+(11+10k_2)s+(6+10k_1)$；
期望 $(s^2+4s+16)(s+10)=s^3+14s^2+56s+160$，得 $K=[15.4,\ 4.5,\ 0.8]$。

In [3]:
a = np.array([[0, 1, 0], [0, 0, 1], [-6, -11, -6]], float)
b = np.array([[0], [0], [10]], float)
x0 = np.array([1.0, 0, 0])
print("rank ctrb(a,b) =", np.linalg.matrix_rank(ctrb(a, b)), " n =", a.shape[0])
p = [-2 + 2 * np.sqrt(3) * 1j, -2 - 2 * np.sqrt(3) * 1j, -10]
k = place(a, b, p)
print("K = place(a,b,p) =", clean(k.ravel(), 4))
print("hand calculation  K = [15.4, 4.5, 0.8]")
aa = a - b @ k
print("eig(a-b*k) =", clean(sorted(np.linalg.eigvals(aa), key=lambda z: (z.real, z.imag))))
print("open-loop eig(a) =", clean(sorted(np.linalg.eigvals(a).real)))

t = np.arange(0, 5.001, 0.01)
x_open, x_closed = initial(a, x0, t), initial(aa, x0, t)
fig = make_subplots(rows=1, cols=2, subplot_titles=("open loop  x' = Ax", "closed loop  x' = (A-BK)x"))
for i, col in enumerate(["#2563eb", "#dc2626", "#16a34a"]):
    fig.add_trace(go.Scatter(x=t, y=x_open[:, i], name=f"x{i+1}", line=dict(color=col)), 1, 1)
    fig.add_trace(go.Scatter(x=t, y=x_closed[:, i], name=f"x{i+1}", line=dict(color=col), showlegend=False), 1, 2)
show(fig, "Regulator: response to x(0) = [1, 0, 0]", "t (s)", "states")
print(f"settling: |x(5)| open loop = {np.linalg.norm(x_open[-1]):.4f}, closed loop = {np.linalg.norm(x_closed[-1]):.2e}")

rank ctrb(a,b) = 3  n = 3
K = place(a,b,p) = [15.4, 4.5, 0.8]
hand calculation  K = [15.4, 4.5, 0.8]
eig(a-b*k) = [(-10+0j), (-2-3.4641j), (-2+3.4641j)]
open-loop eig(a) = [-3.0, -2.0, -1.0]


settling: |x(5)| open loop = 0.0345, closed loop = 2.74e-04


## 3. Pole Placement — Type 1 Servo `poleplace.m`

```matlab
a=[0 1 0;0 0 1;0 -2 -3];  b=[0 0 1]';  c=[1 0 0];  d=0;
sys1=ss(a,b,c,d);
sys1tf=tf(sys1)
sys1zpk=zpk(sys1)
damp(sys1)
%controllability test
M=ctrb(a,b);   co=rank(M);   n=length(a);   unco=n-co
z=0.3;  wn=10;  wd=wn*sqrt(1-z^2)
p=[-z*wn+wd*i -z*wn-wd*i -10];
k=place(sys1.a,sys1.b,p);
%Create the feedback system
aa=a-b*k;  bb=b*k(1);  cc=c;  dd=d;
sys2=ss(aa,bb,cc,dd);
sys2tf=tf(sys2)
sys2zpk=zpk(sys2)
damp(sys2)
figure(1); step(sys1); grid
figure(2); [y t x]=step(sys2); plot(t,y); grid
```

受控體 $\dfrac{1}{s(s+1)(s+2)}$ 本身含積分器（type 1）。控制律 $u=-[0\ k_2\ \cdots\ k_n]x+k_1(r-x_1)=-Kx+k_1r$，
閉迴路 $\dot x=(A-BK)x+Bk_1r$。步級輸入下穩態 $u(\infty)=0$、$x_1(\infty)=r$，所以不需要另加積分器就沒有穩態誤差。

手算驗證：$A-BK$ 特徵多項式 $s^3+(3+k_3)s^2+(2+k_2)s+k_1$，期望 $(s^2+6s+100)(s+10)=s^3+16s^2+160s+1000$，
得 $K=[1000,\ 158,\ 13]$。

In [4]:
a = np.array([[0, 1, 0], [0, 0, 1], [0, -2, -3]], float)
b = np.array([[0], [0], [1]], float)
c = np.array([[1.0, 0, 0]])
num1, den1 = signal.ss2tf(a, b, c, np.zeros((1, 1)))
print("sys1tf: num =", clean(num1.ravel()), " den =", clean(den1))
print("sys1 poles =", clean(sorted(np.linalg.eigvals(a).real)))
M = ctrb(a, b)
co = np.linalg.matrix_rank(M)
print(f"rank ctrb = {co}, n = {a.shape[0]}, uncontrollable states = {a.shape[0] - co}")
z, wn = 0.3, 10
wd = wn * np.sqrt(1 - z ** 2)
print(f"wd = {wd:.4f}")
p = [-z * wn + wd * 1j, -z * wn - wd * 1j, -10]
k = place(a, b, p)
print("K =", clean(k.ravel(), 4))
aa, bb = a - b @ k, b * k[0, 0]
num2, den2 = signal.ss2tf(aa, bb, c, np.zeros((1, 1)))
print("sys2tf: num =", clean(num2.ravel(), 3), " den =", clean(den2, 3))
for pole, w, zz in damp(aa):
    print(f"damp(sys2): pole {complex(pole):.4f}  wn = {w:.4f}  zeta = {zz:.4f}")

t = np.linspace(0, 3, 1501)
_, y2 = signal.step((aa, bb, c, np.zeros((1, 1))), T=t)
t1 = np.linspace(0, 10, 1001)
_, y1 = signal.step((a, b, c, np.zeros((1, 1))), T=t1)
fig = make_subplots(rows=1, cols=2, subplot_titles=("open loop sys1 (contains integrator)", "closed loop sys2"))
fig.add_trace(go.Scatter(x=t1, y=y1, name="sys1", line=dict(color="#64748b")), 1, 1)
fig.add_trace(go.Scatter(x=t, y=y2, name="sys2", line=dict(color="#2563eb")), 1, 2)
fig.add_hline(y=1, line_dash="dot", row=1, col=2)
show(fig, "Type 1 servo: unit step response", "t (s)", "y = x1")
ip = int(np.argmax(y2))
print(f"closed loop: final value = {y2[-1]:.4f}, max overshoot = {100 * (y2[ip] - 1):.1f} %, peak time = {t[ip]:.3f} s")
print(f"second-order formulas with z = 0.3, wn = 10: Mp = {100 * np.exp(-z * np.pi / np.sqrt(1 - z * z)):.1f} %, tp = {np.pi / wd:.3f} s"
      "（第三個極點 -10 離主極點不夠遠（實部只差 3.3 倍），實際超越量明顯較小、尖峰時間較晚）")

sys1tf: num = [0.0, 0.0, 0.0, 1.0]  den = [1.0, 3.0, 2.0, 0.0]
sys1 poles = [-2.0, -1.0, 0.0]
rank ctrb = 3, n = 3, uncontrollable states = 0
wd = 9.5394
K = [1000.0, 158.0, 13.0]
sys2tf: num = [0.0, 0.0, 0.0, 1000.0]  den = [1.0, 16.0, 160.0, 1000.0]
damp(sys2): pole -10.0000+0.0000j  wn = 10.0000  zeta = 1.0000
damp(sys2): pole -3.0000-9.5394j  wn = 10.0000  zeta = 0.3000
damp(sys2): pole -3.0000+9.5394j  wn = 10.0000  zeta = 0.3000


closed loop: final value = 1.0001, max overshoot = 22.5 %, peak time = 0.432 s
second-order formulas with z = 0.3, wn = 10: Mp = 37.2 %, tp = 0.329 s（第三個極點 -10 離主極點不夠遠（實部只差 3.3 倍），實際超越量明顯較小、尖峰時間較晚）


## 4. 全階觀測器 `observer.m`

```matlab
a=[0 1 0;0 0 1;-6 -11 -6];  b=[0 0 1]';  c=[1 0 0];  d=0;
ob=obsv(a,c);
rob=rank(ob)
n = length(a)
q=[-2+2*sqrt(3)*i -2-2*sqrt(3)*i -5];
ke=(place(a',c',q))'
```

觀測器 $\dot{\tilde x}=A\tilde x+Bu+K_e(y-C\tilde x)$，估測誤差 $\varepsilon=x-\tilde x$ 滿足 $\dot\varepsilon=(A-K_eC)\varepsilon$。
可觀測（$\operatorname{rank}O=n$）時，利用 $(A-K_eC)^T=A^T-C^TK_e^T$，觀測器增益可以用極點配置求：$K_e^T=\text{place}(A^T,C^T,q)$。

手算驗證：$A-K_eC$ 的特徵多項式 $s^3+(6+k_1)s^2+(11+6k_1+k_2)s+(6+11k_1+6k_2+k_3)$，
期望 $(s^2+4s+16)(s+5)=s^3+9s^2+36s+80$，得 $K_e=[3,\ 7,\ -1]^T$。

原程式只算出 $K_e$；下面另外模擬（新增）：受控體從 $x(0)=[1,0,0]$ 出發、觀測器從 $\tilde x(0)=0$ 出發，看估測值追上真實狀態。

In [5]:
a = np.array([[0, 1, 0], [0, 0, 1], [-6, -11, -6]], float)
b = np.array([[0], [0], [1]], float)
c = np.array([[1.0, 0, 0]])
print("rank obsv(a,c) =", np.linalg.matrix_rank(obsv(a, c)), " n =", a.shape[0])
q = [-2 + 2 * np.sqrt(3) * 1j, -2 - 2 * np.sqrt(3) * 1j, -5]
ke = place(a.T, c.T, q).T
print("Ke = place(a',c',q)' =", clean(ke.ravel(), 4))
print("eig(a-ke*c) =", clean(sorted(np.linalg.eigvals(a - ke @ c), key=lambda z: (z.real, z.imag))))


def plant_obs(t, s):                         # 狀態 [x; x~]，u = sin t（任意已知輸入）
    x, xh = s[:3], s[3:]
    u = np.sin(t)
    dx = a @ x + b.ravel() * u
    dxh = a @ xh + b.ravel() * u + ke.ravel() * (c @ x - c @ xh)
    return np.concatenate([dx, dxh])


t = np.linspace(0, 4, 801)
sol = solve_ivp(plant_obs, (0, 4), [1, 0, 0, 0, 0, 0], t_eval=t, rtol=1e-9, atol=1e-11)
fig = make_subplots(rows=1, cols=3, subplot_titles=("x1", "x2", "x3"))
for i in range(3):
    fig.add_trace(go.Scatter(x=t, y=sol.y[i], name="true x", line=dict(color="#2563eb"), showlegend=i == 0), 1, i + 1)
    fig.add_trace(go.Scatter(x=t, y=sol.y[i + 3], name="estimate x~", line=dict(color="#dc2626", dash="dash"), showlegend=i == 0), 1, i + 1)
show(fig, "Full-order observer: estimate converges to the true state (u = sin t)", "t (s)", None, h=380)
err = np.linalg.norm(sol.y[:3] - sol.y[3:], axis=0)
print(f"|x - x~|: t=0 -> {err[0]:.3f}, t=2 -> {err[400]:.2e}, t=4 -> {err[-1]:.2e}")

rank obsv(a,c) = 3  n = 3
Ke = place(a',c',q)' = [3.0, 7.0, -1.0]
eig(a-ke*c) = [(-5+0j), (-2-3.4641j), (-2+3.4641j)]


|x - x~|: t=0 -> 1.000, t=2 -> 4.59e-02, t=4 -> 8.77e-04


## 5. 觀測器＋狀態回授 `observerregulator.m`

```matlab
a=[0 1 ;20.6 0];  b=[0; 1];  c=[1 0 ];  d=0;
x0=[1 0];
t=0:0.01:5;
u=zeros(1,length(t));
sys=ss(a,b,c,d);
[y,t,x] = lsim(sys,u,t,x0);
figure(1); plot(t,x); grid
co=ctrb(a,b);  rco=rank(co)
n=length(a)
p=[-1.8+2.4*i -1.8-2.4*i];
k=place(a,b,p)
ob=obsv(a,c);  rob=rank(ob)
q=[-8.1 -7.9];
ke=(place(a',c',q))'
rsys=reg(sys,k,ke)
[y1,t,x1] = lsim(rsys,u,t,x0);
figure(2); plot(t,x1); grid
```

受控體是倒單擺的線性化模型（不穩定極點 $\pm\sqrt{20.6}$）。只量得到 $x_1$，用觀測器估出 $\tilde x$ 再回授 $u=-K\tilde x$。
觀測器極點取 $-8.1$、$-7.9$（`place` 不能配置重根，所以把 $-8$ 的重根拆開）。

手算驗證：$A-BK$ 特徵多項式 $s^2+k_2s+(k_1-20.6)$，期望 $(s+1.8)^2+2.4^2=s^2+3.6s+9$，得 $K=[29.6,\ 3.6]$；
$A-K_eC$ 特徵多項式 $s^2+k_{e1}s+(k_{e2}-20.6)$，期望 $(s+8.1)(s+7.9)=s^2+16s+63.99$，得 $K_e=[16,\ 84.59]^T$。

> **編註**：`reg(sys,k,ke)` 傳回的是**補償器本身**（輸入 $y$、輸出 $u$），原程式 `lsim(rsys,u,t,x0)` 模擬的是補償器的狀態，
> 不是閉迴路系統。閉迴路要把補償器接回受控體（MATLAB：`feedback(sys,rsys,+1)`）。這裡直接寫出閉迴路狀態方程式：
> 以 $[x;\ \varepsilon]$ 為狀態，$\dot x=(A-BK)x+BK\varepsilon$、$\dot\varepsilon=(A-K_eC)\varepsilon$，
> 閉迴路極點＝regulator poles ∪ observer poles（分離原理）。

In [6]:
a = np.array([[0, 1], [20.6, 0]])
b = np.array([[0.0], [1]])
c = np.array([[1.0, 0]])
x0 = np.array([1.0, 0])
print("open-loop eig(a) =", clean(sorted(np.linalg.eigvals(a).real)))
print("rank ctrb =", np.linalg.matrix_rank(ctrb(a, b)), " rank obsv =", np.linalg.matrix_rank(obsv(a, c)), " n =", 2)
k = place(a, b, [-1.8 + 2.4j, -1.8 - 2.4j])
ke = place(a.T, c.T, [-8.1, -7.9]).T
print("K =", clean(k.ravel(), 4))
print("Ke =", clean(ke.ravel(), 4))

# 補償器 rsys = reg(sys,k,ke)：x~' = (A - BK - KeC) x~ + Ke y,  u = -K x~
Ar, Br, Cr = a - b @ k - ke @ c, ke, -k
print("compensator rsys: A =", clean(Ar), " B =", clean(Br.ravel()), " C =", clean(Cr.ravel()))
num_r, den_r = signal.ss2tf(Ar, Br, Cr, np.zeros((1, 1)))
print("compensator transfer function u/y =", clean(num_r.ravel(), 3), "/", clean(den_r, 3))

# 閉迴路：狀態 [x; e]，e = x - x~
Acl = np.block([[a - b @ k, b @ k], [np.zeros((2, 2)), a - ke @ c]])
print("closed-loop poles =", clean(sorted(np.linalg.eigvals(Acl), key=lambda z: (z.real, z.imag))))

t = np.arange(0, 5.001, 0.01)
x_open = initial(a, x0, t)
x_obs = initial(Acl, np.concatenate([x0, x0]), t)           # 觀測器從 x~(0) = 0 出發 → e(0) = x(0)
x_full = initial(a - b @ k, x0, t)                           # 對照：直接量到全部狀態
fig = make_subplots(rows=1, cols=2, subplot_titles=("open loop (unstable)", "closed loop, x1 = angle"))
fig.add_trace(go.Scatter(x=t[:150], y=x_open[:150, 0], name="open loop x1", line=dict(color="#64748b")), 1, 1)
fig.add_trace(go.Scatter(x=t, y=x_obs[:, 0], name="with observer", line=dict(color="#2563eb")), 1, 2)
fig.add_trace(go.Scatter(x=t, y=x_full[:, 0], name="full-state feedback", line=dict(color="#16a34a", dash="dash")), 1, 2)
show(fig, "Observer-based regulator: x(0) = [1, 0]", "t (s)", "x1")
print(f"x1(1.5): open loop = {x_open[150, 0]:.1f}; closed loop with observer = {x_obs[150, 0]:.4f}, full-state = {x_full[150, 0]:.4f}")
print(f"x1(5): with observer = {x_obs[-1, 0]:.2e}")

open-loop eig(a) = [-4.5387, 4.5387]
rank ctrb = 2  rank obsv = 2  n = 2
K = [29.6, 3.6]
Ke = [16.0, 84.59]
compensator rsys: A = [[-16.0, 1.0], [-93.59, -3.6]]  B = [16.0, 84.59]  C = [-29.6, -3.6]
compensator transfer function u/y = [0.0, -778.124, -3690.424] / [1.0, 19.6, 151.19]
closed-loop poles = [(-8.1+0j), (-7.9+0j), (-1.8-2.4j), (-1.8+2.4j)]


x1(1.5): open loop = 452.6; closed loop with observer = -0.1757, full-state = -0.0826
x1(5): with observer = 3.95e-04
